<a href="https://colab.research.google.com/github/SHAIKMEHANAZ/code-orbit-intenship/blob/main/Task1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving CodeOrbitTech_Sales_Product_Dataset_500_Rows_10_Columns.xlsx to CodeOrbitTech_Sales_Product_Dataset_500_Rows_10_Columns.xlsx


In [ ]:
import pandas as pd
import numpy as np

# Read the uploaded Excel file
file_name = list(uploaded.keys())[0]

df = pd.read_excel(
    file_name,
    sheet_name="Raw Sales Data"
)

# Display the first 5 rows
df.head()

,Sale_ID,Order_Date,Product_ID,Product_Name,Category,Quantity,Unit_Price_INR,Sales_Amount_INR,City,Sales_Channel
0,S10001,2025-08-08,P009,Notebook,Stationery,12.0,-120,-1440.0,Vijayawada,Retail Store
1,S10002,2025-12-10,P012,Water Bottle,Accessories,10.0,350,3500.0,Tirupati,Online
2,S10003,2026-02-16,P015,Desk Lamp,Furniture,1.0,950,950.0,Kochi,Wholesale
3,S10004,2025-10-04,P004,Keyboard,Electronics,1.0,1100,1100.0,Tirupati,Online
4,S10005,2026-08-26,P008,Desk,Furniture,8.0,11200,89600.0,Bengaluru,Retail Store


In [ ]:
# Number of rows and columns
print("Dataset shape:", df.shape)

# Column names and data types
df.info()

# Count missing values in each column
print("\nMissing values:")
print(df.isnull().sum())

# Count exact duplicate rows
print("\nExact duplicate rows:", df.duplicated().sum())

# Display basic statistics
df.describe(include="all")

Dataset shape: (500, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Sale_ID           500 non-null    object 
 1   Order_Date        500 non-null    object 
 2   Product_ID        500 non-null    object 
 3   Product_Name      500 non-null    object 
 4   Category          481 non-null    object 
 5   Quantity          481 non-null    float64
 6   Unit_Price_INR    500 non-null    object 
 7   Sales_Amount_INR  482 non-null    float64
 8   City              500 non-null    object 
 9   Sales_Channel     500 non-null    object 
dtypes: float64(2), object(8)
memory usage: 39.2+ KB

Missing values:
Sale_ID              0
Order_Date           0
Product_ID           0
Product_Name         0
Category            19
Quantity            19
Unit_Price_INR       0
Sales_Amount_INR    18
City                 0
Sales_Channel        0
dtype:

,Sale_ID,Order_Date,Product_ID,Product_Name,Category,Quantity,Unit_Price_INR,Sales_Amount_INR,City,Sales_Channel
count,500,500,500,500,481,481.000000,500.0,482.000000,500,500
unique,500,360,16,26,8,NaN,88.0,NaN,10,6
top,S10500,2026-03-25,P004,Keyboard,Electronics,NaN,650.0,NaN,Guntur,Wholesale
freq,1,5,43,42,258,NaN,24.0,NaN,63,174
mean,NaN,NaN,NaN,NaN,NaN,6.864865,NaN,62533.941909,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,3.438815,NaN,116392.089519,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,-1440.000000,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,4.000000,NaN,3262.500000,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,7.000000,NaN,10900.000000,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,10.000000,NaN,78225.000000,NaN,NaN


In [ ]:
# Preserve the original data
cleaned_df = df.copy()

print("Original rows:", len(df))
print("Working copy rows:", len(cleaned_df))

Original rows: 500
Working copy rows: 500


In [ ]:
# Remove leading and trailing spaces from text columns
text_columns = [
    "Product_Name",
    "Category",
    "City",
    "Sales_Channel"
]

for col in text_columns:
    cleaned_df[col] = cleaned_df[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

# Standardize category capitalization
cleaned_df["Category"] = (
    cleaned_df["Category"].str.title()
)

# Standardize sales channel capitalization
cleaned_df["Sales_Channel"] = (
    cleaned_df["Sales_Channel"].str.title()
)

print("Text formatting cleaned.")

Text formatting cleaned.


In [ ]:
# Convert mixed date formats into a consistent date format
cleaned_df["Order_Date"] = pd.to_datetime(
    cleaned_df["Order_Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

# Remove currency symbols and commas from unit prices
cleaned_df["Unit_Price_INR"] = pd.to_numeric(
    cleaned_df["Unit_Price_INR"]
        .astype("string")
        .str.replace(r"[₹,\s]", "", regex=True),
    errors="coerce"
)

print("Date and currency formatting completed.")

Date and currency formatting completed.


In [ ]:
# Fill missing categories using known product names
product_category = (
    cleaned_df.dropna(subset=["Category"])
    .drop_duplicates("Product_Name")
    .set_index("Product_Name")["Category"]
    .to_dict()
)

cleaned_df["Category"] = (
    cleaned_df["Category"].fillna(
        cleaned_df["Product_Name"].map(product_category)
    )
)

# Fill remaining missing categories
cleaned_df["Category"] = (
    cleaned_df["Category"].fillna("Unknown")
)

# Fill missing quantities using the median
median_quantity = cleaned_df["Quantity"].median()

cleaned_df["Quantity"] = (
    cleaned_df["Quantity"].fillna(median_quantity)
)

# Recalculate missing sales amounts where possible
missing_amount = cleaned_df["Sales_Amount_INR"].isna()

cleaned_df.loc[missing_amount, "Sales_Amount_INR"] = (
    cleaned_df.loc[missing_amount, "Quantity"]
    * cleaned_df.loc[missing_amount, "Unit_Price_INR"]
)

print("Missing values handled.")
print(cleaned_df.isnull().sum())

Missing values handled.
Sale_ID             0
Order_Date          0
Product_ID          0
Product_Name        0
Category            0
Quantity            0
Unit_Price_INR      0
Sales_Amount_INR    0
City                0
Sales_Channel       0
dtype: int64


In [ ]:
# Identify repeated sales details, excluding Sale_ID
duplicate_columns = [
    col for col in cleaned_df.columns
    if col != "Sale_ID"
]

duplicate_count = cleaned_df.duplicated(
    subset=duplicate_columns
).sum()

print("Duplicate-like records:", duplicate_count)

# Keep the first occurrence of each matching transaction
cleaned_df = cleaned_df.drop_duplicates(
    subset=duplicate_columns,
    keep="first"
)

print("Rows after duplicate removal:", len(cleaned_df))

Duplicate-like records: 10
Rows after duplicate removal: 490


In [ ]:
print("Final dataset shape:", cleaned_df.shape)

print("\nMissing values:")
print(cleaned_df.isnull().sum())

print("\nDuplicate records:",
      cleaned_df.duplicated().sum())

# Check sales amounts against quantity × unit price
valid = cleaned_df[
    ["Quantity", "Unit_Price_INR", "Sales_Amount_INR"]
].notna().all(axis=1)

print("\nSales amount mismatches:",
      (
          cleaned_df.loc[valid, "Sales_Amount_INR"].round(2)
          != (
              cleaned_df.loc[valid, "Quantity"]
              * cleaned_df.loc[valid, "Unit_Price_INR"]
          ).round(2)
      ).sum())

cleaned_df.head(10)

Final dataset shape: (490, 10)

Missing values:
Sale_ID             0
Order_Date          0
Product_ID          0
Product_Name        0
Category            0
Quantity            0
Unit_Price_INR      0
Sales_Amount_INR    0
City                0
Sales_Channel       0
dtype: int64

Duplicate records: 0

Sales amount mismatches: 18


,Sale_ID,Order_Date,Product_ID,Product_Name,Category,Quantity,Unit_Price_INR,Sales_Amount_INR,City,Sales_Channel
0,S10001,2025-08-08,P009,Notebook,Stationery,12.0,-120,-1440.0,Vijayawada,Retail Store
1,S10002,2025-12-10,P012,Water Bottle,Accessories,10.0,350,3500.0,Tirupati,Online
2,S10003,2026-02-16,P015,Desk Lamp,Furniture,1.0,950,950.0,Kochi,Wholesale
3,S10004,2025-10-04,P004,Keyboard,Electronics,1.0,1100,1100.0,Tirupati,Online
4,S10005,2026-08-26,P008,Desk,Furniture,8.0,11200,89600.0,Bengaluru,Retail Store
5,S10006,2025-07-03,P006,Monitor,Electronics,10.0,11900,119000.0,Chennai,Online
6,S10007,2025-12-31,P008,Desk,Furniture,3.0,11200,33600.0,Bengaluru,Retail Store
7,S10008,2025-12-23,P001,Laptop,Electronics,1.0,55200,55200.0,Tirupati,Wholesale
8,S10009,2025-09-06,P002,Smartphone,Electronics,4.0,25100,100400.0,Guntur,Wholesale
9,S10010,2026-08-04,P016,USB Drive,Electronics,12.0,800,9600.0,Mumbai,Retail Store


In [ ]:
output_file = "Cleaned_Sales_Product_Dataset.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    cleaned_df.to_excel(
        writer,
        sheet_name="Cleaned Sales Data",
        index=False
    )

    # Save a record of the cleaning steps
    cleaning_log = pd.DataFrame({
        "Cleaning Step": [
            "Removed leading/trailing spaces",
            "Standardized text capitalization",
            "Standardized date formats",
            "Converted unit prices to numeric",
            "Handled missing categories",
            "Filled missing quantities using median",
            "Recalculated missing sales amounts",
            "Removed duplicate-like transactions"
        ]
    })

    cleaning_log.to_excel(
        writer,
        sheet_name="Cleaning Log",
        index=False
    )

files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Remove the time portion from Order_Date
cleaned_df["Order_Date"] = (
    pd.to_datetime(cleaned_df["Order_Date"])
    .dt.date
)

# Display the dates without time
cleaned_df[["Order_Date"]].head(10)

,Order_Date
0,2025-08-08
1,2025-12-10
2,2026-02-16
3,2025-10-04
4,2026-08-26
5,2025-07-03
6,2025-12-31
7,2025-12-23
8,2025-09-06
9,2026-08-04


In [ ]:
output_file = "Cleaned_Sales_Product_Dataset.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    cleaned_df.to_excel(
        writer,
        sheet_name="Cleaned Sales Data",
        index=False
    )

files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Correct negative unit prices
cleaned_df["Unit_Price_INR"] = (
    cleaned_df["Unit_Price_INR"].abs()
)

print("Negative unit prices:",
      (cleaned_df["Unit_Price_INR"] < 0).sum())

Negative unit prices: 0


In [ ]:
# Recalculate sales amount
cleaned_df["Sales_Amount_INR"] = (
    cleaned_df["Quantity"] *
    cleaned_df["Unit_Price_INR"]
)

print("Negative sales amounts:",
      (cleaned_df["Sales_Amount_INR"] < 0).sum())

Negative sales amounts: 0


In [ ]:
print("Minimum Unit Price:",
      cleaned_df["Unit_Price_INR"].min())

print("Minimum Sales Amount:",
      cleaned_df["Sales_Amount_INR"].min())

display(cleaned_df.head(10))

Minimum Unit Price: 20
Minimum Sales Amount: 40.0


,Sale_ID,Order_Date,Product_ID,Product_Name,Category,Quantity,Unit_Price_INR,Sales_Amount_INR,City,Sales_Channel
0,S10001,2025-08-08,P009,Notebook,Stationery,12.0,120,1440.0,Vijayawada,Retail Store
1,S10002,2025-12-10,P012,Water Bottle,Accessories,10.0,350,3500.0,Tirupati,Online
2,S10003,2026-02-16,P015,Desk Lamp,Furniture,1.0,950,950.0,Kochi,Wholesale
3,S10004,2025-10-04,P004,Keyboard,Electronics,1.0,1100,1100.0,Tirupati,Online
4,S10005,2026-08-26,P008,Desk,Furniture,8.0,11200,89600.0,Bengaluru,Retail Store
5,S10006,2025-07-03,P006,Monitor,Electronics,10.0,11900,119000.0,Chennai,Online
6,S10007,2025-12-31,P008,Desk,Furniture,3.0,11200,33600.0,Bengaluru,Retail Store
7,S10008,2025-12-23,P001,Laptop,Electronics,1.0,55200,55200.0,Tirupati,Wholesale
8,S10009,2025-09-06,P002,Smartphone,Electronics,4.0,25100,100400.0,Guntur,Wholesale
9,S10010,2026-08-04,P016,USB Drive,Electronics,12.0,800,9600.0,Mumbai,Retail Store


In [ ]:
output_file = "Cleaned_Sales_Product_Dataset.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    cleaned_df.to_excel(
        writer,
        sheet_name="Cleaned Sales Data",
        index=False
    )

files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>